#### Please Run in colab

# Setup

### Environment Setup

In [1]:
!pip install instructor

In [2]:
# !cp /content/macro_financial_forecasting/applications/macro_financial_forecasting/data/danidanou_Bloomberg_Financial_News_train /content/
!cd /content
!rm -rf macro_financial_forecasting

In [3]:
!git clone https://github.com/chuanbinp/macro_financial_forecasting.git

Cloning into 'macro_financial_forecasting'...
remote: Enumerating objects: 1867, done.
remote: Counting objects: 100% (683/683), done.
remote: Compressing objects: 100% (233/233), done.
remote: Total 1867 (delta 516), reused 471 (delta 449), pack-reused 1184 (from 2)
Receiving objects: 100% (1867/1867), 39.55 MiB | 17.85 MiB/s, done.
Resolving deltas: 100% (1174/1174), done.


In [4]:
# !cp /content/danidanou_Bloomberg_Financial_News_train /content/macro_financial_forecasting/applications/macro_financial_forecasting/data/danidanou_Bloomberg_Financial_News_train

In [5]:
%cd macro_financial_forecasting/applications/macro_financial_forecasting/src

/content/macro_financial_forecasting/applications/macro_financial_forecasting/src


### Mount GDrive

In [6]:
from google.colab import drive

# This will prompt you to authorize Colab to access your Google Drive.
drive.mount('/content/gdrive')
GDRIVE_PATH = "/content/gdrive/MyDrive/macro_financial_forecasting_output/"

Mounted at /content/gdrive


### Code Setup

In [7]:
from config import Config
from train_data_loader import TrainDataLoader
from data_model.bloomberg_news_entry import BloombergNewsEntry
from google.colab import userdata
import os

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
config = Config("../config.env")

train_data_loader = TrainDataLoader(config)

# Train Data Loader

In [8]:
print("Starting loading pipeline ...")
print(f"Config: {config}")

train_ds = train_data_loader.load()
print("Loading pipeline completed.")

Starting loading pipeline ...
Config: Config(
  gemini_api_key: !secret!
  openai_api_key: !secret!
  llm_model: openai/gpt-5-nano-2025-08-07
  industries: ['Information Technology', 'Health Care', 'Financials', 'Consumer Discretionary', 'Communication Services', 'Industrials', 'Consumer Staples', 'Energy', 'Utilities', 'Real Estate', 'Materials', 'General Market', 'None']
  dataset_name: danidanou/Bloomberg_Financial_News
  dataset_dir: ../data/
  rss_feeds: ['https://feeds.bloomberg.com/news/news.rss', 'https://feeds.bloomberg.com/markets/news.rss', 'https://feeds.bloomberg.com/business/news.rss', 'https://feeds.bloomberg.com/technology/news.rss', 'https://feeds.bloomberg.com/politics/news.rss', 'https://feeds.bloomberg.com/wealth/news.rss', 'https://feeds.bloomberg.com/economics/news.rss', 'https://feeds.bloomberg.com/green/news.rss', 'https://feeds.bloomberg.com/pursuits/news.rss', 'https://feeds.bloomberg.com/opinion/news.rss', 'https://feeds.bloomberg.com/finance/news.rss', 'http

bloomberg_financial_data.parquet.gzip:   0%|          | 0.00/482M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/446762 [00:00<?, ? examples/s]


--- Download Successful! ---


Map:   0%|          | 0/446762 [00:00<?, ? examples/s]

Training dataset processed.

--- Starting validation of 446762 entries ---


Validating entries: 100%|██████████| 446762/446762 [00:30<00:00, 14626.23it/s]


--- Validation Complete! ---
Training dataset validated.
Saving processed dataset to local cache at '../data/danidanou_Bloomberg_Financial_News_train'...
Total number of rows: 446762
Loading pipeline completed.


# Data Processing Pipeline

Update these 2 variable to specify which indices to process.

In [9]:
DATA_START=0
DATA_END=100

In [ ]:
from processor import NewsProcessor
import nest_asyncio
nest_asyncio.apply()

processor = NewsProcessor(config)
train_ds = processor.remove_redundant_info(train_ds[DATA_START:DATA_END])
df = processor.enrich_news_entries_with_classifications(train_ds, save_path=f"{GDRIVE_PATH}processed_news") #Sample size
df = processor.group_by_date_and_industry(df, save_path=f"{GDRIVE_PATH}grouped_news")
df = processor.filter_and_analyze_news(df)
df = processor.extract_impactful_news(df, top_n=3, save_path=f"{GDRIVE_PATH}impact_news")
df = processor.get_consolidated_sentiment(df, save_path=f"{GDRIVE_PATH}sentiment_news")

tokenizer_config.json:   0%|          | 0.00/252 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/758 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

In [ ]:
df = processor.get_explanation(df, save_path="sentiment_news")

,Headline,Date,Link,Article,SentimentScore,Industry
0,"Ivory Coast Keeps Cocoa Export Tax Below 22%, ...",2011-10-06,http://www.bloomberg.com/news/2011-10-06/ivory...,"Export taxes on cocoa beans from Ivory Coast ,...",0.057765,Financials
1,USDA Boxed Beef Cutout Closing Prices for Octo...,2011-10-06,http://www.bloomberg.com/news/2011-10-06/usda-...,October 6 (Bloomberg) -- This table details bo...,0.857337,Materials
2,U.S. September Small Business Jobs Summary,2011-10-06,http://www.bloomberg.com/news/2011-10-06/u-s-s...,U.S. small business plans to hire declined in ...,0.597226,None
3,Greece’s GSEE Says Won’t Meet For Talks With T...,2011-10-06,http://www.bloomberg.com/news/2011-10-06/greec...,"Greece ’s biggest private sector union group, ...",0.793129,Financials
4,Clean-Tech Companies Should Get 10-Year Tax Br...,2011-10-06,http://www.bloomberg.com/news/2011-10-06/clean...,"Reed Hundt, head of the Coalition for Green Ca...",0.019132,Energy
...,...,...,...,...,...,...
9995,Dendreon Gains After Aetna Expands Coverage of...,2012-09-28,http://www.bloomberg.com/news/2012-09-28/dendr...,"Dendreon Corp. (DNDN) , maker of the prostate ...",0.463558,Health Care
9996,USDA Illinois Soybean Crush Report for Sept. 28,2012-09-28,http://www.bloomberg.com/news/2012-09-28/usda-...,"Springfield, Illinois , Sept. 28 (Bloomberg Da...",0.918252,Consumer Discretionary
9997,Obama Probably Will Combine Agencies in Second...,2012-09-28,http://www.bloomberg.com/news/2012-09-28/obama...,President Barack Obama will probably revisit h...,-0.043081,Energy
9998,Hungarian Regulator Reviewing Banks Setting In...,2012-09-28,http://www.bloomberg.com/news/2012-09-28/hunga...,Hungary’s financial regulator PSZAF is conduct...,0.862027,Financials


In [ ]:
df